# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suyashverma0/flyrank-ml-workspace2/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [6]:
import os
import subprocess
import pandas as pd
from pathlib import Path

# Step 1: Clone repository
repo_path = Path("/content/flyrank-ml-workspace2")

if not repo_path.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/suyashverma0/flyrank-ml-workspace2.git",
            str(repo_path)
        ],
        check=True
    )
else:
    print("Repository already exists.")

# Step 2: Locate dataset
dataset_name = "content_refresh_anonymized.csv"

matches = list(repo_path.rglob(dataset_name))

# Step 3: Load dataset if found
if matches:
    data_path = matches[0]
    print("Dataset found:", data_path)

    df = pd.read_csv(data_path)

    print("\nDataset successfully loaded!")
    print("Shape:", df.shape)
    display(df.head())

else:
    print("Dataset not found in GitHub repository.")
    print("We need to retrieve it from the FlyRank Hugging Face dataset.")

Dataset found: /content/flyrank-ml-workspace2/data/raw/content_refresh_anonymized.csv

Dataset successfully loaded!
Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [8]:
import pandas as pd
import numpy as np

# Create target
df["is_declining_proxy"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

# Select features
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent"
]

feature_columns = numeric_features + categorical_features

# Create X, y and groups
X = df[feature_columns].copy()
y = df["is_declining_proxy"].copy()
groups = df["client_id"].copy()

# Convert numeric columns
for col in numeric_features:
    X[col] = pd.to_numeric(X[col], errors="coerce")

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Groups:", groups.nunique())
print("Section 1 variables created successfully!")

X shape: (30000, 10)
y shape: (30000,)
Groups: 32
Section 1 variables created successfully!


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

# Verify client separation
train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])

assert train_clients.isdisjoint(test_clients)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Client overlap:", len(train_clients.intersection(test_clients)))
print("Train target rate:", round(y_train.mean(), 4))
print("Test target rate:", round(y_test.mean(), 4))

Training rows: 22885
Testing rows: 7115
Training clients: 24
Testing clients: 8
Client overlap: 0
Train target rate: 0.55
Test target rate: 0.5165


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# 1. Preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# 2. Train model
model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)

# 3. Model probabilities
model_probabilities = model.predict_proba(X_test)[:, 1]

# 4. Prepare baseline columns
baseline_columns = [
    "impressions_90d",
    "sessions_90d",
    "ctr",
    "engagement_rate",
    "days_since_last_update",
    "avg_position"
]

for col in baseline_columns:
    train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
    test_df[col] = pd.to_numeric(test_df[col], errors="coerce")

# 5. Training-only thresholds
ctr_cutoff = train_df.loc[
    train_df["impressions_90d"] >= 300, "ctr"
].quantile(0.25)

engagement_cutoff = train_df.loc[
    train_df["sessions_90d"] > 0, "engagement_rate"
].quantile(0.25)

# 6. Baseline rules on test data
visible = test_df["impressions_90d"] >= 300
has_sessions = test_df["sessions_90d"] > 0
has_position = test_df["avg_position"] > 0

low_ctr = visible & (test_df["ctr"] <= ctr_cutoff)

low_engagement = (
    has_sessions &
    (test_df["engagement_rate"] <= engagement_cutoff)
)

stale = test_df["days_since_last_update"] >= 180

weak_position = (
    visible &
    has_position &
    (test_df["avg_position"] > 20)
)

test_df["baseline_score"] = (
    low_ctr.astype(int) +
    low_engagement.astype(int) +
    stale.astype(int) +
    weak_position.astype(int)
)

# 7. Same test rows for both rankings
baseline_order = test_df.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
).index

model_order = pd.Series(
    model_probabilities,
    index=X_test.index
).sort_values(ascending=False).index

# 8. Precision@K
def precision_at_k(order, labels, k):
    selected = labels.loc[order[:min(k, len(order))]]
    return selected.mean()

results = []

for k in [10, 50, 100]:
    results.append({
        "Metric": f"Precision@{k}",
        "Week-4 Baseline": round(
            precision_at_k(baseline_order, y_test, k), 4
        ),
        "Logistic Regression": round(
            precision_at_k(model_order, y_test, k), 4
        )
    })

comparison = pd.DataFrame(results)

print("Training CTR cutoff:", ctr_cutoff)
print("Training engagement cutoff:", engagement_cutoff)
print("Test base rate:", round(y_test.mean(), 4))

display(comparison)

Training CTR cutoff: 0.05
Training engagement cutoff: 0.0
Test base rate: 0.5165


,Metric,Week-4 Baseline,Logistic Regression
0,Precision@10,0.50,0.60
1,Precision@50,0.56,0.64
2,Precision@100,0.59,0.55


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import classification_report, confusion_matrix

# 1. Predictions
predictions = (model_probabilities >= 0.5).astype(int)

print("Classification Report:")
print(classification_report(
    y_test,
    predictions,
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

# 2. Feature importance through coefficients
feature_names = model.named_steps[
    "preprocess"
].get_feature_names_out()

coefficients = model.named_steps[
    "classifier"
].coef_[0]

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

print("\nStrongest Positive Features:")
display(
    importance_df.sort_values(
        "Coefficient", ascending=False
    ).head(10)
)

print("\nStrongest Negative Features:")
display(
    importance_df.sort_values(
        "Coefficient", ascending=True
    ).head(10)
)

# 3. Error analysis
error_df = test_df[[
    "content_type",
    "content_age_days",
    "days_since_last_update"
]].copy()

error_df["actual"] = y_test
error_df["predicted"] = predictions
error_df["probability"] = model_probabilities

error_df["error_type"] = np.select(
    [
        (error_df["predicted"] == 1) & (error_df["actual"] == 0),
        (error_df["predicted"] == 0) & (error_df["actual"] == 1)
    ],
    ["False Positive", "False Negative"],
    default="Correct"
)

print("\nError Distribution:")
display(
    error_df["error_type"].value_counts().to_frame("Count")
)

print("\nFalse Positives:")
display(
    error_df[error_df["error_type"] == "False Positive"].head(5)
)

print("\nFalse Negatives:")
display(
    error_df[error_df["error_type"] == "False Negative"].head(5)
)

Classification Report:
              precision    recall  f1-score   support

           0       0.52      0.42      0.46      3440
           1       0.54      0.64      0.58      3675

    accuracy                           0.53      7115
   macro avg       0.53      0.53      0.52      7115
weighted avg       0.53      0.53      0.53      7115

Confusion Matrix:
[[1445 1995]
 [1335 2340]]

Strongest Positive Features:


,Feature,Coefficient
15,cat__competition_level_Unknown,1.278390
20,cat__main_intent_informational,0.478443
19,cat__main_intent_commercial,0.346472
22,cat__main_intent_transactional,0.251073
6,num__days_since_last_update,0.236519
16,cat__content_type_feedly article,0.169688
3,num__word_count,0.153551
0,num__search_volume,-0.001789
2,num__cpc,-0.005097
1,num__competition,-0.048247



Strongest Negative Features:


,Feature,Coefficient
21,cat__main_intent_navigational,-0.558978
13,cat__competition_level_LOW,-0.515314
18,cat__main_intent_Unknown,-0.452851
12,cat__competition_level_HIGH,-0.352979
14,cat__competition_level_MEDIUM,-0.345939
5,num__content_age_days,-0.321616
8,num__missingindicator_competition,-0.243376
7,num__missingindicator_search_volume,-0.243376
9,num__missingindicator_cpc,-0.243376
4,num__char_count,-0.203062



Error Distribution:


,Count
error_type,
Correct,3785
False Positive,1995
False Negative,1335



False Positives:


,content_type,content_age_days,days_since_last_update,actual,predicted,probability,error_type
13,keyword article,238,103,0,1,0.656875,False Positive
34,comparison article,140,8,0,1,0.603025,False Positive
36,keyword article,187,20,0,1,0.575878,False Positive
56,keyword article,145,20,0,1,0.601815,False Positive
64,keyword article,106,8,0,1,0.598494,False Positive



False Negatives:


,content_type,content_age_days,days_since_last_update,actual,predicted,probability,error_type
1,keyword article,445,25,1,0,0.419664,False Negative
23,keyword article,502,20,1,0,0.316664,False Negative
54,keyword article,502,22,1,0,0.319031,False Negative
58,keyword article,494,20,1,0,0.271486,False Negative
81,keyword article,502,22,1,0,0.319031,False Negative


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.